In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+



In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+



In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.nyc_taxi")
display(dbutils.fs.ls("/Volumes/workspace/default/nyc_taxi"))

[]

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/nyc_taxi"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/nyc_taxi/taxi_zone_lookup.csv,taxi_zone_lookup.csv,12331,1795494057000
dbfs:/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2023-01.parquet,yellow_tripdata_2023-01.parquet,47673370,1795493800000
dbfs:/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2023-02.parquet,yellow_tripdata_2023-02.parquet,47748012,1795493800000
dbfs:/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2023-03.parquet,yellow_tripdata_2023-03.parquet,56127762,1795493834000
dbfs:/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2023-04.parquet,yellow_tripdata_2023-04.parquet,54222699,1795493828000
dbfs:/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2023-05.parquet,yellow_tripdata_2023-05.parquet,58654627,1795493845000
dbfs:/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2023-06.parquet,yellow_tripdata_2023-06.parquet,54999465,1795493830000
dbfs:/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2023-07.parquet,yellow_tripdata_2023-07.parquet,48361828,1795493804000
dbfs:/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2023-08.parquet,yellow_tripdata_2023-08.parquet,48152353,1795493803000
dbfs:/Volumes/workspace/default/nyc_taxi/yellow_tripdata_2023-09.parquet,yellow_tripdata_2023-09.parquet,47895515,1795493803000


In [0]:
from pyspark.sql import functions as F

DATA_DIR = "/Volumes/workspace/default/nyc_taxi"


files = dbutils.fs.ls(DATA_DIR)

taxi_paths = sorted(
    f.path
    for f in files
    if f.name.startswith("yellow_tripdata_")
    and f.name.endswith(".parquet")
)

total_bytes = sum(
    f.size
    for f in files
    if f.name.startswith("yellow_tripdata_")
    and f.name.endswith(".parquet")
)

print(f"Monthly files: {len(taxi_paths)}")
print(f"Total size: {total_bytes / (1024**3):.2f} GiB")


monthly_dfs = []

for path in taxi_paths:
    month_df = spark.read.parquet(path)

    normalized = month_df.select(
        F.col("tpep_pickup_datetime")
         .cast("timestamp_ntz")
         .alias("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime")
         .cast("timestamp_ntz")
         .alias("tpep_dropoff_datetime"),
        F.col("passenger_count").cast("double").alias("passenger_count"),
        F.col("trip_distance").cast("double").alias("trip_distance"),
        F.col("PULocationID").cast("long").alias("PULocationID"),
        F.col("DOLocationID").cast("long").alias("DOLocationID"),
        F.col("payment_type").cast("long").alias("payment_type"),
        F.col("fare_amount").cast("double").alias("fare_amount"),
        F.col("tip_amount").cast("double").alias("tip_amount"),
        F.col("total_amount").cast("double").alias("total_amount"),
    )

    monthly_dfs.append(normalized)


trips_raw = monthly_dfs[0]

for month_df in monthly_dfs[1:]:
    trips_raw = trips_raw.unionByName(month_df)

print(f"Monthly DataFrames combined: {len(monthly_dfs)}")
trips_raw.printSchema()


ZONE_PATH = f"{DATA_DIR}/taxi_zone_lookup.csv"

zones_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ZONE_PATH)
)

zones_raw.printSchema()

Monthly files: 24
Total size: 1.24 GiB
Monthly DataFrames combined: 24
root
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [0]:
selected_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
]


trips_filtered = (
    trips_raw
    .select(*selected_cols)
    .filter(
        F.col("tpep_pickup_datetime").isNotNull()
        & F.col("tpep_dropoff_datetime").isNotNull()
    )
    .filter(F.col("trip_distance").between(0.1, 100))
    .filter(
        F.col("fare_amount").between(1, 500)
        & F.col("total_amount").between(1, 600)
        & F.col("passenger_count").between(1, 6)
    )
    .withColumn(
        "trip_duration_minutes",
        F.expr(
            "timestampdiff("
            "SECOND, tpep_pickup_datetime, tpep_dropoff_datetime"
            ") / 60.0"
        ),
    )
    .filter(F.col("trip_duration_minutes").between(1, 180))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_year", F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn(
        "tip_pct",
        F.when(
            F.col("fare_amount") > 0,
            F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2),
        ),
    )
)


pickup_zones = zones_raw.select(
    F.col("LocationID").cast("long").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone"),
)


enriched = trips_filtered.join(
    F.broadcast(pickup_zones),
    on="PULocationID",
    how="left",
)


display(
    enriched.select(
        "pickup_date",
        "pickup_hour",
        "pickup_borough",
        "pickup_zone",
        "trip_distance",
        "trip_duration_minutes",
        "total_amount",
        "tip_pct",
    ).limit(10)
)

pickup_date,pickup_hour,pickup_borough,pickup_zone,trip_distance,trip_duration_minutes,total_amount,tip_pct
2023-01-01,0,Manhattan,Midtown Center,0.97,8.433333,14.3,0.0
2023-01-01,0,Manhattan,Central Park,1.1,6.316667,16.9,50.63
2023-01-01,0,Manhattan,Clinton East,2.51,12.750000,34.9,100.67
2023-01-01,0,Manhattan,Gramercy,1.43,10.833333,19.68,28.77
2023-01-01,0,Manhattan,Midtown Center,1.84,12.300000,27.8,78.13
2023-01-01,0,Manhattan,Upper West Side South,1.66,10.450000,20.52,28.26
2023-01-01,0,Manhattan,Lincoln Square East,11.7,22.733333,64.44,23.5
2023-01-01,0,Manhattan,Midtown South,2.95,14.933333,28.38,32.09
2023-01-01,0,Manhattan,Lenox Hill West,3.01,10.900000,19.9,0.0
2023-01-01,0,Manhattan,Union Sq,1.8,8.733333,19.68,28.77


In [0]:
enriched = enriched.filter(
    (F.col("tpep_pickup_datetime") >= F.lit("2023-01-01").cast("timestamp_ntz"))
    & (F.col("tpep_pickup_datetime") < F.lit("2025-01-01").cast("timestamp_ntz"))
)


monthly_summary = (
    enriched
    .groupBy("pickup_year", "pickup_month", "pickup_borough")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("total_charges"),
        F.avg("total_amount").alias("avg_total_amount"),
        F.avg("trip_distance").alias("avg_distance_miles"),
        F.avg("trip_duration_minutes").alias("avg_duration_minutes"),
    )
)


display(
    monthly_summary.orderBy(
        "pickup_year", "pickup_month", "pickup_borough"
    )
)

pickup_year,pickup_month,pickup_borough,trip_count,total_charges,avg_total_amount,avg_distance_miles,avg_duration_minutes
2023,1,Bronx,2817,95785.47000000042,34.00265175718865,7.03149804756833,30.4043900046
2023,1,Brooklyn,13050,412247.44000000064,31.58984214559392,5.201131800766279,23.2415044733
2023,1,EWR,22,2686.5400000000004,122.11545454545457,11.479999999999999,21.7810605909
2023,1,Manhattan,2552439,5.771529150999731E7,22.61182011009756,2.3958384118091782,12.6628845977
2023,1,N/A,307,24052.110000000015,78.34563517915315,8.712410423452765,19.7610206189
2023,1,Queens,265481,1.8750496610016514E7,70.62839378342146,12.910748038466249,31.5067633210
2023,1,Staten Island,209,15474.950000000008,74.04282296650722,18.03043062200957,47.8373205646
2023,1,Unknown,36984,1457827.389999994,39.41778579926439,5.708812729829073,18.6664999291
2023,2,Bronx,2793,95793.09000000024,34.29756176154681,6.820082348728958,30.6850877197
2023,2,Brooklyn,11834,373294.1099999994,31.54420398850764,5.178225452087206,23.4525195789


In [0]:
monthly_summary.createOrReplaceTempView("monthly_taxi_summary")
enriched.createOrReplaceTempView("taxi_trips")

sql_yearly_comparison = spark.sql("""
    WITH yearly AS (
        SELECT
            pickup_borough,
            pickup_year,
            SUM(trip_count) AS annual_trips,
            SUM(total_charges) AS annual_total_charges
        FROM monthly_taxi_summary
        WHERE pickup_borough IN (
            'Manhattan', 'Queens', 'Brooklyn', 'Bronx', 'Staten Island'
        )
        GROUP BY pickup_borough, pickup_year
    ),
    comparison AS (
        SELECT
            *,
            LAG(annual_trips) OVER (
                PARTITION BY pickup_borough
                ORDER BY pickup_year
            ) AS previous_year_trips
        FROM yearly
    )
    SELECT
        pickup_borough,
        pickup_year,
        annual_trips,
        ROUND(annual_total_charges, 2) AS annual_total_charges,
        ROUND(
            100.0 * (annual_trips - previous_year_trips)
            / previous_year_trips,
            2
        ) AS trip_growth_pct
    FROM comparison
    ORDER BY pickup_borough, pickup_year
""")

display(sql_yearly_comparison)

pickup_borough,pickup_year,annual_trips,annual_total_charges,trip_growth_pct
Bronx,2023,47311,1614529.35,null
Bronx,2024,66794,2635801.19,41.18
Brooklyn,2023,179725,5828665.66,null
Brooklyn,2024,295089,9616891.56,64.19
Manhattan,2023,31425326,7.5044744816E8,null
Manhattan,2024,31538106,7.5315922307E8,0.36
Queens,2023,3470818,2.5569027048E8,null
Queens,2024,3441807,2.5807659484E8,-0.84
Staten Island,2023,1347,104159.38,null
Staten Island,2024,238,19141.39,-82.33


In [0]:
sql_peak_hours = spark.sql("""
    WITH hourly AS (
        SELECT
            pickup_borough,
            pickup_hour,
            COUNT(*) AS trip_count,
            AVG(trip_duration_minutes) AS avg_duration_minutes,
            AVG(total_amount) AS avg_total_amount
        FROM taxi_trips
        WHERE pickup_year = 2024
          AND pickup_borough IN (
              'Manhattan', 'Queens', 'Brooklyn', 'Bronx', 'Staten Island'
          )
        GROUP BY pickup_borough, pickup_hour
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY pickup_borough
                ORDER BY trip_count DESC, pickup_hour
            ) AS hour_rank
        FROM hourly
    )
    SELECT
        pickup_borough,
        hour_rank,
        pickup_hour,
        trip_count,
        ROUND(avg_duration_minutes, 2) AS avg_duration_minutes,
        ROUND(avg_total_amount, 2) AS avg_total_amount
    FROM ranked
    WHERE hour_rank <= 3
    ORDER BY pickup_borough, hour_rank
""")

display(sql_peak_hours)

pickup_borough,hour_rank,pickup_hour,trip_count,avg_duration_minutes,avg_total_amount
Bronx,1,9,6304,47.54,37.77
Bronx,2,8,5976,50.05,38.2
Bronx,3,10,5594,46.80,38.39
Brooklyn,1,8,23357,42.08,34.15
Brooklyn,2,9,22996,39.66,33.76
Brooklyn,3,10,20782,37.99,32.19
Manhattan,1,18,2312921,14.23,24.06
Manhattan,2,17,2206022,15.58,25.2
Manhattan,3,19,2007438,13.37,23.52
Queens,1,16,250650,48.65,81.31


In [0]:
from time import perf_counter

OUTPUT_DIR = f"{DATA_DIR}/outputs"
CLEAN_PATH = f"{OUTPUT_DIR}/cleaned_trips"

start = perf_counter()

(
    enriched
    .repartition(24, "pickup_year", "pickup_month")
    .write
    .mode("overwrite")
    .partitionBy("pickup_year", "pickup_month")
    .parquet(CLEAN_PATH)
)

print(f"Cleaned Parquet written in {perf_counter() - start:.2f} seconds")

for name, result in [
    ("monthly_summary", monthly_summary),
    ("yearly_comparison", sql_yearly_comparison),
    ("peak_hours", sql_peak_hours),
]:
    result.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/{name}")
    print(f"Saved: {name}")

display(dbutils.fs.ls(OUTPUT_DIR))
display(dbutils.fs.ls(f"{CLEAN_PATH}/pickup_year=2024"))

Cleaned Parquet written in 25.48 seconds
Saved: monthly_summary
Saved: yearly_comparison
Saved: peak_hours


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/,cleaned_trips/,0,1795544841286
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/monthly_summary/,monthly_summary/,0,1795544841286
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/peak_hours/,peak_hours/,0,1795544841286
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/yearly_comparison/,yearly_comparison/,0,1795544841286


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=1/,pickup_month=1/,0,1795544843001
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=10/,pickup_month=10/,0,1795544843001
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=11/,pickup_month=11/,0,1795544843001
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=12/,pickup_month=12/,0,1795544843001
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=2/,pickup_month=2/,0,1795544843001
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=3/,pickup_month=3/,0,1795544843001
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=4/,pickup_month=4/,0,1795544843001
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=5/,pickup_month=5/,0,1795544843001
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=6/,pickup_month=6/,0,1795544843001
dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips/pickup_year=2024/pickup_month=7/,pickup_month=7/,0,1795544843001


In [0]:
cleaned_trips = spark.read.parquet(CLEAN_PATH)

january_query = (
    cleaned_trips
    .filter(
        (F.col("pickup_year") == 2024)
        & (F.col("pickup_month") == 1)
    )
    .filter(F.col("trip_distance") > 2)
    .groupBy("pickup_borough")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("total_amount").alias("avg_total_amount"),
    )
)

january_query.explain(mode="formatted")

start = perf_counter()
january_results = january_query.collect()
print(f"Query completed in {perf_counter() - start:.2f} seconds")

for row in january_results:
    print(row)

== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [5]: [trip_distance#16936, total_amount#16941, pickup_borough#16946, pickup_year#16948, pickup_month#16949]
DictionaryFilters: [(trip_distance#16936 > 2.0)]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/nyc_taxi/outputs/cleaned_trips]
PartitionFilters: [isnotnull(pickup_year#16948), isnotnull(pickup_month#16949), (pickup_year#16948 = 2024), (pickup_month#16949 = 1)]
ReadSchema: struct<trip_distance:double,total_amount:double,pickup_borough:string>
RequiredDataFilters: [isnotnull(trip_distance#16936), (trip_distance#16936 > 2.0

Spark transformations such as filter(), withColumn(), and select() are lazy: they construct a logical execution plan without immediately processing all rows. The count() action triggers execution and returns the number of matching trips to Python. In this example, it returned 1,126,477 trips. Because counting rows does not require amount_per_mile, Spark can prune that calculation from the executed plan.

In [0]:
lazy_demo = (
    cleaned_trips
    .filter(
        (F.col("pickup_year") == 2024)
        & (F.col("pickup_month") == 1)
    )
    .filter(F.col("trip_distance") > 2)
    .withColumn(
        "amount_per_mile",
        F.col("total_amount") / F.col("trip_distance"),
    )
    .select("pickup_borough", "amount_per_mile")
)

print("Transformations defined; no action has been called.")

start = perf_counter()
demo_count = lazy_demo.count()

print(f"Action completed: {demo_count:,} matching trips")
print(f"count() elapsed time: {perf_counter() - start:.2f} seconds")

Transformations defined; no action has been called.
Action completed: 1,126,477 matching trips
count() elapsed time: 0.64 seconds


## Performance analysis

The pipeline processes 24 monthly Yellow Taxi Parquet files from 2023–2024 totaling 1.24 GiB. Each file is read with its own schema, numeric fields are normalized, and the resulting DataFrames are combined using unionByName without a shuffle. Selecting required columns and filtering invalid distances, fares, passenger counts, durations, and dates before downstream aggregation reduces the data carried through the pipeline. The small zone lookup is broadcast for the left join, avoiding repartitioning the large trip table for that join. Aggregations still require a shuffle; partial aggregation reduces the volume exchanged.

The cleaned output is partitioned by pickup_year and pickup_month. The January 2024 query profile explicitly shows partition filters pickup_year = 2024 and pickup_month = 1, and scan-level filters trip_distance IS NOT NULL and trip_distance > 2. Only trip_distance, total_amount, and pickup_borough are required from the Parquet data; year and month come from partition attributes. The profile reports one file and one partition read, 9.29 MB read, 2,711,747 rows scanned, and 1,126,477 output rows after filtering. It returned eight borough groups in 897 ms server wall-clock time; the Python collect timer reported 1.11 seconds including client overhead.

The measured bottleneck was storage access: the scan took 512 ms, including 509 ms of cloud storage request duration, while partial grouping took 26 ms. No disk spilling was reported, and 94% of aggregated task time was in Photon. Repartitioning before the output write is an intentional shuffle to organize files by month; the cleaned write took 25.48 seconds. Other shuffles remain necessary for groupBy and SQL window functions. These observations describe individual runs, not a controlled before/after speedup benchmark; different queries and caching conditions cannot be compared as equivalent workloads.

## Key findings and interpretation

- In the cleaned sample, Manhattan trips increased from 31,425,326 in 2023 to 31,538,106 in 2024 (+0.36%); Queens declined 0.84%.
- Brooklyn increased 64.19% from a much smaller base (179,725 to 295,089 trips). These changes describe retained Yellow Taxi records, not all transportation demand.
- The busiest 2024 pickup hour was 18:00–18:59 in Manhattan, 16:00–16:59 in Queens, 08:00–08:59 in Brooklyn, and 09:00–09:59 in the Bronx. Staten Island has very few observations and should be interpreted cautiously.
- total_amount includes passenger charges such as taxes and tips and is not driver net income. Missing passenger counts and records outside the cleaning thresholds are excluded, so conclusions apply to this filtered population.

## Data source and reproducibility

Official source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

Inputs: yellow_tripdata_2023-01.parquet through yellow_tripdata_2024-12.parquet, plus taxi_zone_lookup.csv. Upload these into /Volumes/workspace/default/nyc_taxi and run the notebook from top to bottom. Results are written under outputs/cleaned_trips, outputs/monthly_summary, outputs/yearly_comparison, and outputs/peak_hours. Re-running the output cell replaces these generated results.
